# What Is Actually Making Sofia's Air Dirty?
## A Multi-Predictor Regression Study of PM10 in Sofia, Bulgaria

---

Sofia consistently ranks among the most polluted EU capitals. Every winter, a thick grey smog settles over the city — a visible sign of dangerously high concentrations of **particulate matter (PM)**. The most regulated size fraction is **PM10** — particles under 10 micrometres in diameter — for which the EU sets a legal annual average limit of 40 µg/m³. Sofia routinely exceeds it.

The usual suspects are well known: wood-burning stoves, old diesel cars, industrial activity. But Bulgaria's electricity grid has also changed dramatically — lignite (brown coal) generation collapsed 59% in 2024, with CO₂-free sources now comprising 66% of the power mix.

So here is the question nobody has cleanly answered: **does grid-level decarbonization actually show up in Sofia's air quality (AQ), or is the signal completely buried under heating season and weather?**

To answer it, we can't just plot energy mix against pollution and call it correlation. We have to control for the things that dominate — temperature, wind, the heating season — and then look at what's left. That remainder, if anything, belongs to the grid.

We will:
1. Collect and merge three independent data sources: EEA station-level PM10 measurements, Ember monthly electricity generation data, and ERA5 meteorological reanalysis via Open-Meteo
2. Explore the data and establish the seasonal and meteorological structure of Sofia's PM10 exceedances
3. Build a multiple linear regression model with heating season, wind speed, temperature, and lignite generation share as predictors
4. Quantify the partial effect of the energy mix after controlling for the dominant confounders
5. Cross-validate findings using a Random Forest model and compare feature importances
6. Interpret the result honestly — including if the grid signal doesn't survive the controls

---
## Section 1 — Problem Formulation

### 1.1 The Air Quality Problem

Air pollution in Sofia is fundamentally a **source attribution problem**. PM10 in the air above the city at any given moment is the sum of contributions from multiple independent sources: domestic solid fuel heating, road traffic, industry, and — indirectly — the national electricity grid through its fossil fuel plants.

The challenge is that these sources are correlated with each other and with external factors. Cold days mean more heating *and* more atmospheric stability (reduced vertical mixing, so pollutants accumulate). Winter means both heating season *and* low wind speeds in Sofia's valley basin. A naive regression of PM10 on energy mix would absorb all of these.

The central question is:

> **To what extent does the electricity generation mix influence PM10 levels in Sofia, once we control for heating season and meteorological conditions?**

This is a **causal inference problem cast as a regression problem**. We cannot run a controlled experiment — we can't turn lignite generation on and off while holding everything else constant. What we *can* do is partial out the known confounders and examine whether the energy mix variable retains a statistically significant coefficient in the residual.

### 1.2 Why Multiple Regression?

Multiple linear regression (MLR) is the right starting tool here because it directly answers a question of the form: *what is the effect of variable X on outcome Y, holding variables Z₁, Z₂, ... constant?* The coefficient on lignite share is exactly that partial effect — the change in PM10 associated with a one-unit change in lignite generation share, after removing the contribution of temperature, wind, and season.

This is not the only approach. The tradeoffs between methods are:

| Method | Pros | Cons |
|---|---|---|
| Multiple Linear Regression | Interpretable coefficients, p-values, partial effects | Assumes linearity, sensitive to outliers |
| Random Forest | Captures non-linear interactions, robust | No direct partial effects, black box |
| ARIMA / time series | Handles autocorrelation explicitly | Doesn't naturally incorporate exogenous variables |
| Causal inference (IV, DiD) | Strongest causal claims | Requires valid instruments, complex setup |

We use MLR as the primary method and Random Forest as a cross-validation tool. Both are run on the same feature set, so we can compare their feature importance rankings directly.

### 1.3 Hypotheses

**H₁ (Primary):** Heating season and meteorological variables will explain the majority of variance in daily PM10 concentrations, with lignite generation share retaining a small but statistically significant positive partial effect after controlling for these factors.

**H₂ (Alternative):** The partial effect of lignite generation on PM10 is not statistically significant once heating season and meteorology are controlled for — implying grid-level decarbonization has not produced a detectable signal in urban air quality at the monthly timescale.

**H₀ (Null):** None of the predictor variables has a significant linear relationship with PM10 concentrations in Sofia.

### 1.4 Assumptions & Constraints

Our model makes the following explicit assumptions:
- ERA5 reanalysis data from Open-Meteo is a valid proxy for local meteorological conditions in Sofia (~9 km spatial resolution)
- National-level generation data from Ember is a valid proxy for grid dispatch conditions affecting Sofia — Bulgaria operates as a single bidding zone with no sub-regional breakdown
- Domestic heating intensity is proxied by a binary heating season variable (October–March) and heating degree days derived from temperature — not directly measured
- The relationship between predictors and PM10 is approximately linear at monthly aggregation

**Known limitations:**
- The model cannot directly measure domestic wood/coal burning — the largest single PM10 source in Sofia. This is the primary structural limitation.
- Traffic counts are not included. Day-of-week features partially capture this.
- Causal claims cannot be made from observational regression alone. This is an associational study with controlled confounders.